# YOLOv8n Solar Panel Training + LicheeRV Nano Simulation
### Solar PV Anomaly Detection — Image Inference Pipeline

**Pipeline stages:**
- **Stage 1** — Train YOLOv8n on solar panel dataset
- **Stage 2** — Export to ONNX INT8 (SG2002 NPU format)
- **Stage 3** — Validate per-class metrics
- **Stage 4** — LicheeRV Nano simulation benchmark
- **Stage 5** — Confidence score extraction → Logistic Regression input

**Class mapping:** `0=bird_drop  1=cracked  2=dusty  3=panel`

## 0 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install ultralytics onnx onnxruntime opencv-python psutil pyyaml pandas --quiet

## 1 — Imports

In [ ]:
import csv, gc, os, sys, time
from pathlib import Path
import cv2
import numpy as np
import psutil
import matplotlib
matplotlib.use('Agg')  # Avoid font cache errors
import matplotlib.pyplot as plt
print('Imports OK')

## 2 — Configuration

> **Edit `DATASET_YAML` below before running anything.**

In [ ]:
# ── EDIT THIS ─────────────────────────────────────────────────────────
DATASET_YAML = '/home/pc/solar_panels_train/ultralytics_new20250625/datasets/datasets_3000/data.yaml'

# ── Base directory of your project ────────────────────────────────────
BASE_DIR = Path('/home/Monke/Documents/Project/RTU/Datasets/RTU-MODEL')
# ───────────────────────────────────────────────────────────────────────

CONFIG = {
    'dataset_yaml':  DATASET_YAML,
    'model_base':    'yolov8n.pt',
    'epochs':        100,
    'imgsz':         640,
    'batch':         16,
    'patience':      20,
    # Absolute path stops Ultralytics nesting under runs/detect/
    'project':       str(BASE_DIR / 'runs' / 'solar_yolo'),
    'name':          'train',
    'device':        0,
    'workers':       4,
    'optimizer':     'AdamW',
    'lr0':           0.001,
    'lrf':           0.01,
    'weight_decay':  0.0005,
    'mosaic':        1.0,
    'mixup':         0.1,
    'export_format':   'onnx',
    'export_int8':     True,
    'export_simplify': True,
    'sim_memory_ceiling_mb': 200,
    'sim_benchmark_frames':  100,
    'sim_target_latency_ms': 2000,
    'class_names':   {0: 'bird_drop', 1: 'cracked', 2: 'dusty', 3: 'panel'},
    'score_columns': {0: 'img_bird_drop_score', 1: 'img_cracked_score',
                      2: 'img_dusty_score',     3: 'img_panel_score'},
    'conf_threshold': 0.25,
}

# ── Auto-detect latest best.pt ─────────────────────────────────────────
# Searches correct path AND the old nested runs/detect/ path from previous runs
def find_latest_weights(base_dir):
    search_roots = [
        base_dir / 'runs' / 'solar_yolo',
        base_dir / 'runs' / 'detect' / 'runs' / 'solar_yolo',
    ]
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates += list(root.rglob('best.pt'))
    if not candidates:
        return None
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    return str(latest)

weights_path = find_latest_weights(BASE_DIR)
onnx_path    = None

print(f'Dataset yaml:    {DATASET_YAML}')
print(f'Yaml exists:     {os.path.exists(DATASET_YAML)}')
print(f'Latest weights:  {weights_path}')
print(f'Weights exist:   {weights_path is not None and os.path.exists(str(weights_path))}')

## Stage 1 — Train YOLOv8n

If `best.pt` was already found above, **this cell skips training automatically.**  
Delete the weights folder only if you want to retrain from scratch.

In [ ]:
def stage_train():
    from ultralytics import YOLO

    if not os.path.exists(CONFIG['dataset_yaml']):
        print(f"ERROR: data.yaml not found at {CONFIG['dataset_yaml']}")
        return None

    # Check new correct path
    best_existing = find_latest_weights(BASE_DIR)
    if best_existing:
        print(f'Found existing weights: {best_existing}')
        print('Skipping training. Delete weights folder to retrain.')
        return best_existing

    print(f'Dataset:  {CONFIG["dataset_yaml"]}')
    print(f'Project:  {CONFIG["project"]}')
    print(f'Epochs:   {CONFIG["epochs"]} (patience={CONFIG["patience"]})')
    print(f'Batch:    {CONFIG["batch"]}  |  Imgsz: {CONFIG["imgsz"]}')

    model   = YOLO(CONFIG['model_base'])
    results = model.train(
        data         = CONFIG['dataset_yaml'],
        epochs       = CONFIG['epochs'],
        imgsz        = CONFIG['imgsz'],
        batch        = CONFIG['batch'],
        patience     = CONFIG['patience'],
        project      = CONFIG['project'],
        name         = CONFIG['name'],
        device       = CONFIG['device'],
        workers      = CONFIG['workers'],
        optimizer    = CONFIG['optimizer'],
        lr0          = CONFIG['lr0'],
        lrf          = CONFIG['lrf'],
        weight_decay = CONFIG['weight_decay'],
        mosaic       = CONFIG['mosaic'],
        mixup        = CONFIG['mixup'],
        verbose      = True,
        plots        = False,  # Disabled — matplotlib font cache workaround
        save         = True,
        save_period  = 10,
    )

    best = find_latest_weights(BASE_DIR)
    print(f'Training complete. Best weights: {best}')
    print('\n=== FINAL METRICS ===')
    for k, v in results.results_dict.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
    return best

weights_path = stage_train()
print(f'\nweights_path = {weights_path}')

## Stage 2 — Export to ONNX INT8

Converts trained weights → ONNX INT8 format for SG2002 NPU.

In [ ]:
def stage_export(weights_path=None):
    from ultralytics import YOLO

    if weights_path is None:
        weights_path = find_latest_weights(BASE_DIR)
    if not weights_path or not os.path.exists(weights_path):
        print('ERROR: No weights found. Run Stage 1 first.')
        return None

    print(f'Weights:  {weights_path}')
    print('Format:   ONNX INT8 — SG2002 NPU format')

    model = YOLO(weights_path)
    export_path = model.export(
        format   = CONFIG['export_format'],
        imgsz    = CONFIG['imgsz'],
        int8     = CONFIG['export_int8'],
        simplify = CONFIG['export_simplify'],
        dynamic  = False,
    )

    pt_mb   = os.path.getsize(weights_path) / (1024*1024)
    onnx_mb = os.path.getsize(export_path)  / (1024*1024)
    print(f'\nPyTorch:    {pt_mb:.2f} MB')
    print(f'ONNX INT8:  {onnx_mb:.2f} MB')
    print(f'Reduction:  {(1 - onnx_mb/pt_mb)*100:.1f}%')
    print(f'Output:     {export_path}')
    return export_path

onnx_path = stage_export(weights_path)
print(f'\nonnx_path = {onnx_path}')

## Stage 3 — Validate Per-class Metrics

AP50 < 0.5 on any class = that class needs more training data.

In [ ]:
def stage_validate(weights_path=None):
    from ultralytics import YOLO

    if weights_path is None:
        weights_path = find_latest_weights(BASE_DIR)
    if not weights_path or not os.path.exists(weights_path):
        print('ERROR: No weights found.')
        return

    print(f'Validating: {weights_path}')
    model   = YOLO(weights_path)
    metrics = model.val(
        data    = CONFIG['dataset_yaml'],
        imgsz   = CONFIG['imgsz'],
        device  = CONFIG['device'],
        verbose = True,
        plots   = False,
    )

    print('\n=== PER-CLASS METRICS ===')
    for i, name in CONFIG['class_names'].items():
        try:
            p  = metrics.box.p[i]
            r  = metrics.box.r[i]
            f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
            ap = metrics.box.ap[i]
            flag = '' if ap >= 0.5 else '  <- NEEDS MORE DATA'
            print(f'  {name:12s}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={ap:.3f}{flag}')
        except Exception:
            print(f'  {name:12s}  metrics unavailable')
    print(f'\n  mAP50:    {metrics.box.map50:.4f}')
    print(f'  mAP50-95: {metrics.box.map:.4f}')

stage_validate(weights_path)

## Stage 4 — LicheeRV Nano Simulation Benchmark

- CPU single-thread inference (mirrors NPU single-core operation)
- 200MB memory ceiling (256MB DDR3 minus OS)
- NPU latency estimated at 15× CPU speedup

In [ ]:
def stage_benchmark(onnx_path=None, test_images_dir=None):
    import onnxruntime as ort

    if onnx_path is None:
        # Search for any .onnx in both paths
        search_roots = [
            BASE_DIR / 'runs' / 'solar_yolo',
            BASE_DIR / 'runs' / 'detect' / 'runs' / 'solar_yolo',
        ]
        candidates = []
        for root in search_roots:
            if root.exists():
                candidates += list(root.rglob('*.onnx'))
        if not candidates:
            print('ERROR: No ONNX file found. Run Stage 2 first.')
            return None
        onnx_path = str(max(candidates, key=lambda p: p.stat().st_mtime))

    print(f'ONNX:           {onnx_path}')
    print(f'Memory ceiling: {CONFIG["sim_memory_ceiling_mb"]} MB')
    print(f'Target latency: {CONFIG["sim_target_latency_ms"]} ms')

    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.inter_op_num_threads = 1
    session = ort.InferenceSession(onnx_path, sess_options=opts,
                                   providers=['CPUExecutionProvider'])
    input_name  = session.get_inputs()[0].name
    input_shape = session.get_inputs()[0].shape
    print(f'Model input:    {input_name}  shape={input_shape}')

    test_frames = []
    if test_images_dir and os.path.exists(test_images_dir):
        img_paths = [p for p in Path(test_images_dir).rglob('*')
                     if p.suffix.lower() in ['.jpg','.jpeg','.png']]
        img_paths = img_paths[:CONFIG['sim_benchmark_frames']]
        for p in img_paths:
            img = cv2.imread(str(p))
            if img is not None:
                img = cv2.resize(img, (CONFIG['imgsz'], CONFIG['imgsz']))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
                img = np.expand_dims(np.transpose(img,(2,0,1)),0)
                test_frames.append(img)
    while len(test_frames) < CONFIG['sim_benchmark_frames']:
        test_frames.append(np.random.rand(1,3,CONFIG['imgsz'],CONFIG['imgsz']).astype(np.float32))

    proc = psutil.Process(os.getpid())
    gc.collect()
    mem_start = proc.memory_info().rss / (1024*1024)
    for f in test_frames[:5]:
        session.run(None, {input_name: f})
    gc.collect()

    print(f'Benchmarking {len(test_frames)} frames...')
    latencies = []
    for i, frame in enumerate(test_frames):
        t0 = time.perf_counter()
        session.run(None, {input_name: frame})
        latencies.append((time.perf_counter()-t0)*1000)
        if (i+1) % 25 == 0:
            print(f'  [{i+1:3d}/{len(test_frames)}] {latencies[-1]:.1f}ms')

    mem_peak  = proc.memory_info().rss / (1024*1024)
    latencies = np.array(latencies)
    onnx_mb   = os.path.getsize(onnx_path) / (1024*1024)
    mem_total = mem_peak - mem_start
    mean_lat  = np.mean(latencies)
    est_npu   = mean_lat / 15.0
    mem_ok    = mem_total < CONFIG['sim_memory_ceiling_mb']
    lat_ok    = est_npu   < CONFIG['sim_target_latency_ms']

    report = f'''
╔══════════════════════════════════════════════════════╗
║     LicheeRV Nano (SG2002) Simulation Report        ║
╠══════════════════════════════════════════════════════╣
║  MODEL                                               ║
║    ONNX INT8 size:       {onnx_mb:>8.2f} MB             ║
║    Fits in 200MB:        {"YES" if onnx_mb < 200 else "NO":>8s}                 ║
╠══════════════════════════════════════════════════════╣
║  CPU BENCHMARK                                       ║
║    Mean latency:         {mean_lat:>8.1f} ms             ║
║    Median latency:       {np.median(latencies):>8.1f} ms             ║
║    P95 latency:          {np.percentile(latencies,95):>8.1f} ms             ║
╠══════════════════════════════════════════════════════╣
║  NPU ESTIMATE (1 TOPS, 15x CPU speedup)              ║
║    Estimated NPU:        {est_npu:>8.1f} ms             ║
║    Target:               {CONFIG["sim_target_latency_ms"]:>8d} ms             ║
║    Meets target:         {"YES" if lat_ok else "NO":>8s}                 ║
╠══════════════════════════════════════════════════════╣
║  MEMORY                                              ║
║    Total used:           {mem_total:>8.1f} MB             ║
║    Remaining:            {CONFIG["sim_memory_ceiling_mb"]-mem_total:>8.1f} MB             ║
║    Memory safe:          {"YES" if mem_ok else "NO":>8s}                 ║
╠══════════════════════════════════════════════════════╣
║  VERDICT: {"VIABLE FOR DEPLOYMENT" if mem_ok and lat_ok else "NEEDS REVIEW":>42s}  ║
╚══════════════════════════════════════════════════════╝
    '''
    print(report)
    report_path = str(BASE_DIR / 'benchmark_report.txt')
    with open(report_path, 'w') as f:
        f.write(report)
    print(f'Saved: {report_path}')
    return onnx_path

test_dir = None
try:
    import yaml
    with open(CONFIG['dataset_yaml']) as f:
        d = yaml.safe_load(f)
    test_dir = d.get('test')
except Exception:
    pass

onnx_path = stage_benchmark(onnx_path, test_dir)

## Stage 5 — Confidence Score Extraction

Produces `[bird_drop, cracked, dusty, panel]` score vectors — the exact output  
the RTU sends to the Fog layer for logistic regression input.

In [ ]:
def extract_scores(session, image_path):
    scores = {col: 0.0 for col in CONFIG['score_columns'].values()}
    scores['img_panel_score'] = 1.0

    img = cv2.imread(image_path)
    if img is None:
        return scores

    img = cv2.resize(img, (CONFIG['imgsz'], CONFIG['imgsz']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = np.expand_dims(np.transpose(img, (2,0,1)), 0)

    input_name  = session.get_inputs()[0].name
    outputs     = session.run(None, {input_name: img})
    predictions = outputs[0]

    if len(predictions.shape) == 3:
        predictions = predictions[0].T
    if predictions.shape[1] > 4:
        class_confs = predictions[:, 4:]
        max_confs   = class_confs.max(axis=0)
        detected    = False
        for cls_id, conf in enumerate(max_confs):
            if cls_id in CONFIG['score_columns'] and conf >= CONFIG['conf_threshold']:
                scores[CONFIG['score_columns'][cls_id]] = float(conf)
                if cls_id != 3:
                    detected = True
        if detected:
            anomaly = max(scores['img_bird_drop_score'],
                          scores['img_cracked_score'],
                          scores['img_dusty_score'])
            scores['img_panel_score'] = max(0.0, 1.0 - anomaly)
    total = sum(scores.values())
    if total > 0:
        scores = {k: v/total for k, v in scores.items()}
    return scores


def stage_infer(onnx_path=None, image_path=None, test_dir=None):
    import onnxruntime as ort, pandas as pd

    if onnx_path is None:
        search_roots = [
            BASE_DIR / 'runs' / 'solar_yolo',
            BASE_DIR / 'runs' / 'detect' / 'runs' / 'solar_yolo',
        ]
        candidates = []
        for root in search_roots:
            if root.exists():
                candidates += list(root.rglob('*.onnx'))
        if not candidates:
            print('ERROR: No ONNX found. Run Stage 2 first.')
            return
        onnx_path = str(max(candidates, key=lambda p: p.stat().st_mtime))

    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

    if image_path and os.path.exists(image_path):
        scores   = extract_scores(session, image_path)
        dominant = max(scores, key=scores.get).replace('img_','').replace('_score','')
        print('Score vector:')
        for k, v in scores.items():
            marker = ' <-' if k == max(scores, key=scores.get) else ''
            print(f'  {k}: {v:.4f}{marker}')
        print(f'Dominant: {dominant}')

    elif test_dir and os.path.exists(test_dir):
        img_paths = [p for p in Path(test_dir).rglob('*')
                     if p.suffix.lower() in ['.jpg','.jpeg','.png']]
        print(f'Processing {len(img_paths)} images...')
        results = []
        for i, p in enumerate(img_paths):
            s = extract_scores(session, str(p))
            dominant = max(s, key=s.get).replace('img_','').replace('_score','')
            results.append({'image': p.name, 'dominant': dominant, **s})
            if (i+1) % 50 == 0:
                print(f'  [{i+1}/{len(img_paths)}]')

        out_csv = str(BASE_DIR / 'sample_inference_scores.csv')
        with open(out_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
            writer.writeheader()
            writer.writerows(results)
        print(f'Saved: {out_csv}')

        df = pd.DataFrame(results)
        print('\n=== CLASS DISTRIBUTION ===')
        print(df['dominant'].value_counts())
        score_cols = [c for c in df.columns if c.startswith('img_')]
        print('\n=== MEAN SCORES PER CLASS ===')
        print(df.groupby('dominant')[score_cols].mean().round(3))
    else:
        print('Set image_path or test_dir and rerun.')

### Run inference — set your path below

In [ ]:
# Single image
# stage_infer(onnx_path, image_path='/path/to/panel.jpg')

# Batch — auto-detect test dir from data.yaml
test_dir = None
try:
    import yaml
    with open(CONFIG['dataset_yaml']) as f:
        d = yaml.safe_load(f)
    test_dir = d.get('test')
except Exception:
    pass

if test_dir:
    stage_infer(onnx_path, test_dir=test_dir)
else:
    print('No test dir in data.yaml — set image_path manually above.')

## Summary — Output Files

In [ ]:
print('=== OUTPUT FILES ===')
w = find_latest_weights(BASE_DIR)
onnx_candidates = []
for root in [BASE_DIR/'runs'/'solar_yolo', BASE_DIR/'runs'/'detect'/'runs'/'solar_yolo']:
    if root.exists():
        onnx_candidates += list(root.rglob('*.onnx'))
o = str(max(onnx_candidates, key=lambda p: p.stat().st_mtime)) if onnx_candidates else None

files = [
    (w,                                            'PyTorch weights'),
    (o,                                            'ONNX INT8 for LicheeRV Nano'),
    (str(BASE_DIR / 'benchmark_report.txt'),       'Simulation report'),
    (str(BASE_DIR / 'sample_inference_scores.csv'),'Score vectors from test set'),
]
for path, desc in files:
    if path and os.path.exists(path):
        size   = f'{os.path.getsize(path)/(1024*1024):.2f} MB'
        status = 'OK'
    else:
        size, status = '', 'MISSING'
    print(f'  [{status:7s}] {desc:35s} {path or "not found"}')